# Prediction |error| distribution (PANAV test set, 10-seed ensemble)

Histogram and KDE of the pooled per-residue |error|. Residues: GT present, prediction mask True, no proline.

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np
import pandas as pd

PRED = "../../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json"
GT   = "../../../data/rBMRB/cluster_30p/PANAV/test.json"

def set_style(base=12, family="Arial"):
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base + 2, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })

## 1. Load predictions and ground truth

In [ ]:
with open(PRED) as f: pred_data = json.load(f)
with open(GT)   as f: gt_data   = json.load(f)

# 10-seed ensemble: use the already-averaged per-residue "mean" (paired with "mask"), not the
# per-seed "pred" matrix
pred_dict, mask_dict = pred_data["mean"], pred_data["mask"]
dictionary = {v: k for k, v in gt_data["entry_ID"].items()}
print("Predictions:", len(pred_dict), "entries")

## 2. Per-residue |error|

In [ ]:
records = []
length_mismatch = 0
for entry in gt_data["entry_ID"].values():
    key = dictionary.get(entry)
    if key is None or entry not in pred_dict:
        continue

    gt_seq, gt_shifts = gt_data["sequence"][key], gt_data["N"][key]
    pred_arr_full = np.array(pred_dict[entry])
    pred_mask_full = np.array(mask_dict[entry], dtype=bool)
    n = len(gt_seq)
    # predictions are right-padded to the batch's longest sequence -- truncate to n instead of
    # skipping the whole protein when longer; only skip if actually shorter than gt_seq
    if len(pred_arr_full) < n or len(pred_mask_full) < n:
        length_mismatch += 1
        continue
    pred_arr = pred_arr_full[:n]
    pred_mask = pred_mask_full[:n]

    gt_mask_1 = np.array([res != "P" for res in gt_seq])
    none_mask = np.array([g is not None for g in gt_shifts])
    mask = gt_mask_1 & none_mask & pred_mask
    if mask.sum() == 0:
        continue

    full_indices = np.where(mask)[0]
    pred_f = pred_arr[mask]
    gt_f   = np.array(gt_shifts)[mask].astype(float)
    abs_error = np.abs(pred_f - gt_f)
    error = pred_f - gt_f

    for i, full_idx in enumerate(full_indices):
        records.append({
            "entry": entry,
            "resnum": int(full_idx) + 1,
            "residue_type": gt_seq[full_idx],
            "abs_error": float(abs_error[i]),
            "error": float(error[i]),
        })

df = pd.DataFrame(records)
print(f"Total residues: {len(df)}  |  proteins: {df.entry.nunique()}  |  length mismatch skipped: {length_mismatch}")

## 3. Summary statistics

In [ ]:
x = df["abs_error"].to_numpy()
stats = {
    "n": len(x), "mean": x.mean(), "median": np.median(x), "std": x.std(),
    "p10": np.percentile(x, 10), "p50": np.percentile(x, 50), "p90": np.percentile(x, 90),
    "p95": np.percentile(x, 95), "p99.5": np.percentile(x, 99.5), "max": x.max(),
}
for k, v in stats.items():
    print(f"{k:>6s}: {v:.4f}" if k != "n" else f"{k:>6s}: {v}")

## 4. Distribution plot

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

set_style(base=14)

color_hist = "#61AFEF"
color_kde  = "#E06C75"

XMAX = np.percentile(x, 100)   # trim the extreme tail for a readable x-range
bins = np.linspace(0, XMAX, 61)

fig, ax1 = plt.subplots(figsize=(8, 5.5))
ax2 = ax1.twinx()

ax1.hist(x, bins=bins, color=color_hist, alpha=0.55, edgecolor="white", linewidth=0.4,
          label="raw counts")
ax1.set_xlabel("|error| (ppm)")
ax1.set_ylabel("count", color=color_hist)
ax1.tick_params(axis="y", labelcolor=color_hist)
ax1.set_xlim(0, XMAX)

kde = gaussian_kde(x)
xs = np.linspace(0, XMAX, 400)
ax2.plot(xs, kde(xs), color=color_kde, lw=2.4, label="KDE")
ax2.set_ylabel("density", color=color_kde)
ax2.tick_params(axis="y", labelcolor=color_kde)
ax2.set_ylim(bottom=0)

ax1.axvline(stats["median"], color="k", lw=1.2, ls="--", alpha=0.7,
            label=f"median = {stats['median']:.3f} ppm")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, frameon=False, loc="upper right")

fig.tight_layout()
fig.savefig("Figs/error_distribution_panav_10seedens.svg", bbox_inches="tight")
fig.savefig("Figs/error_distribution_panav_10seedens.png", dpi=300, bbox_inches="tight")
print("saved -> Figs/error_distribution_panav_10seedens.(svg|png)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

set_style(base=14)

color_hist = "#61AFEF"
color_kde  = "#E06C75"

XMAX = np.percentile(x, 99.5)   # trim the extreme tail for a readable x-range
bins = np.linspace(0, XMAX, 61)

fig, ax1 = plt.subplots(figsize=(8, 5.5))
ax2 = ax1.twinx()

ax1.hist(x, bins=bins, color=color_hist, alpha=0.55, edgecolor="white", linewidth=0.4,
          label="raw counts")
ax1.set_xlabel("|error| (ppm)")
ax1.set_ylabel("count", color=color_hist)
ax1.tick_params(axis="y", labelcolor=color_hist)
ax1.set_xlim(0, XMAX)

kde = gaussian_kde(x)
xs = np.linspace(0, XMAX, 400)
ax2.plot(xs, kde(xs), color=color_kde, lw=2.4, label="KDE")
ax2.set_ylabel("density", color=color_kde)
ax2.tick_params(axis="y", labelcolor=color_kde)
ax2.set_ylim(bottom=0)

ax1.axvline(stats["median"], color="k", lw=1.2, ls="--", alpha=0.7,
            label=f"median = {stats['median']:.3f} ppm")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, frameon=False, loc="upper right")

fig.tight_layout()
fig.savefig("Figs/error_distribution_panav_10seedens_part99p5.svg", bbox_inches="tight")
fig.savefig("Figs/error_distribution_panav_10seedens_part99p5.png", dpi=300, bbox_inches="tight")
print("saved -> Figs/error_distribution_panav_10seedens.(svg|png)")
plt.show()

In [ ]:
x = df["error"].to_numpy()
stats = {
    "n": len(x), "mean": x.mean(), "median": np.median(x), "std": x.std(),
    "p05": np.percentile(x, 5), "p50": np.percentile(x, 50), "p95": np.percentile(x, 95),
    "min": x.min(), "max": x.max(),
}
for k, v in stats.items():
    print(f"{k:>6s}: {v:.4f}" if k != "n" else f"{k:>6s}: {v}")

import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

set_style(base=14)

color_hist = "#61AFEF"
color_kde  = "#E06C75"

XMAX = np.percentile(np.abs(x), 99.5)   # symmetric trim on |error|, keeps both tails readable
bins = np.linspace(-XMAX, XMAX, 61)

fig, ax1 = plt.subplots(figsize=(8, 5.5))
ax2 = ax1.twinx()

ax1.hist(x, bins=bins, color=color_hist, alpha=0.55, edgecolor="white", linewidth=0.4,
          label="raw counts")
ax1.set_xlabel("error = pred - gt (ppm)")
ax1.set_ylabel("count", color=color_hist)
ax1.tick_params(axis="y", labelcolor=color_hist)
ax1.set_xlim(-XMAX, XMAX)

kde = gaussian_kde(x)
xs = np.linspace(-XMAX, XMAX, 400)
ax2.plot(xs, kde(xs), color=color_kde, lw=2.4, label="KDE")
ax2.set_ylabel("density", color=color_kde)
ax2.tick_params(axis="y", labelcolor=color_kde)
ax2.set_ylim(bottom=0)

ax1.axvline(0, color="grey", lw=1, ls=":", alpha=0.8, label="0")
ax1.axvline(stats["median"], color="k", lw=1.2, ls="--", alpha=0.7,
            label=f"median = {stats['median']:.3f} ppm")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, frameon=False, loc="upper right")

fig.tight_layout()
fig.savefig("Figs/error_distribution_panav_10seedens_signed.svg", bbox_inches="tight")
fig.savefig("Figs/error_distribution_panav_10seedens_signed.png", dpi=300, bbox_inches="tight")
print("saved -> Figs/error_distribution_panav_10seedens_signed.(svg|png)")
plt.show()